**Mount Google Drive**

In [ ]:
# Mount the drive
from google.colab import drive
drive.mount('/content/drive')

# Change directory
import os
os.chdir("/content/drive/MyDrive")

**List Drive Folders & Files**

In [ ]:
# List content of MyDrive
!ls "/content/drive/MyDrive/"

# XGBoost
Tree boosting is highly effective and widely used machine learning method. XGBoost stands for “Extreme Gradient Boosting”, where the term “Gradient Boosting” originates from the paper Greedy Function Approximation: A Gradient Boosting Machine, by Friedman.

XGBoost, which is used widely by data scientists to achieve state-of-the-art results on many machine learning challenges. XGBoost initially started as a research project by Tianqi Chen as part of the Distributed (Deep) Machine Learning Community (DMLC) group. Initially, it began as a terminal application which could be configured using a libsvm configuration file.

XGBoost was designed to be used with large, complicated datasets and is one of the most popular machine learning algorithm to deal with structured data. It is an advance version of gradient boosting method that is designed to focus on computational speed and model efficiency. XGBoost is preferred over other tree based model as it supports

* Parallelization
* Distributed computing methods
* Out-of-core computing
* Cache optimization


The algorithm is highly customizable and accurate in its predictions as it seeks to minimise the objective function

$$Obj^{(t)}=\underbrace{\sum_{i=1}^n l \,(y_i,\widetilde{y}_i^{(t)})}_{\text{training loss}} \quad + \underbrace{\sum_{i=1}^t\Omega(f_i)}_{\text{regularisation term}}$$


$$Obj^{(t)}=
{\sum_{i=1}^n l \,(y_i,\widetilde{y}_i^{(t-1)}+f_t(x_i))} + {\sum_{i=1}^t\Omega(f_i)}$$

The first term (over all instances) measures the distance between the true label and the output from the model while the second term (over all trees) penalises models that are too complex. Optimising loss tend to create more complex models while optimizing regularisation tends to generalise simplier models.

In XGBoost, we define the complexity as

$$\Omega(f) = \gamma T + \frac{1}{2} \lambda {w_j}^2$$

where,

* $T$ is the number of terminal nodes, or leaves in a tree
* $\gamma$ and $\lambda$ are regularisation terms which is a user definable penalty for pruning
> `gamma`: minimum loss reduction required to make a further partition on a leaf node of the tree. Large gamma will lead to more conservative algorithm. <br>
> `lambda`: L2 regularization term on weights. Increasing this value will make model more conservative. Normalised to number of training examples.
* ${w_j}^2$ is the score of each leaf


While metrics for regression is straight forward, it can be complex for classification (logistic loss) where we use Taylor expansion of the loss function up to the second order. After removing all the constants, the specific objective (function) at step $t$ becomes

$$
\sum_{i=1}^{n}[g_i f_t(x_i) + \frac{1}{2} h_i f_t^{2}(x_i)] + \Omega(f_t)
$$

where the $g_i$ and $h_i$ are defined as

$$
g_i = \partial_{\tilde{y_i}^{(t-1)}} l(y_i, \tilde{y_i}^{(t-1)}) \\
h_i = \partial^2_{\tilde{y_i}^{(t-1)}} l(y_i, \tilde{y_i}^{(t-1)})
$$

Refer [here](https://xgboost.readthedocs.io/en/latest/tutorials/model.html) for further details.

Manipulating the above formulation for classification, the leaf output is given as

$$
\frac{\sum R_i}{\sum [P_i*(1 - P_i)] + \color{red}\lambda}
$$

where, $R$ is the residual; $P_i$ is the previous probability; $\lambda$ is the regularization parameter.

---

_**Note:** Some third-party Python libraries (e.g., yfinance, quantmod) depend on external data sources that may be restricted or inaccessible in certain regions, including China. Users in these regions are advised to consult their local instructor or substitute with locally supported alternatives._

---


**Install Packages**

In [ ]:
# Install packages
!pip install -q xgboost wandb pyfolio-reloaded

**Import Libraries**

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt

# Classifier
from xgboost import XGBClassifier, plot_importance, to_graphviz

# Preprocessing
from sklearn.model_selection import (train_test_split,
                                    TimeSeriesSplit,
                                    cross_val_score
                                    )

from sklearn.utils.class_weight import compute_sample_weight

# Metrics
from sklearn.metrics import (accuracy_score,
                             balanced_accuracy_score,
                             RocCurveDisplay,
                             ConfusionMatrixDisplay,
                             PrecisionRecallDisplay,
                             classification_report
                            )

from sklearn.metrics import (roc_auc_score,
                             f1_score,
                             precision_score,
                             recall_score
                             )

## Section 1: Experiment Tracking

We use Weights & Biases (**W&B**), which integrates seamlessly with Python workflows, enabling real-time logging, run comparison, and artifact management. It brings discipline and visibility to experimentation, making iteration faster and results more reliable.

In [ ]:
# Experiment Tracker
# import os
# Remove lingering sweep variables from the notebook environment
# os.environ.pop("WANDB_SWEEP_ID", None)
# os.environ.pop("WANDB_RUN_ID", None)

import wandb
from google.colab import userdata

# Log in to Weights & Biases
wandb.login(key=userdata.get('WANDB_API_KEY'))


## Section 2: The workflow

We'll employ XGBoost classifier from `scikit-learn` for stock / equity index trend prediction.


| Steps        | Workflow                  | Remarks                                                         |
|:-------------|:--------------------------|:----------------------------------------------------------------|
|Step 1        | Ideation                  | Define objective, success metrics     |
|Step 2        | Data Collection           | Gather and integrate data
|Step 3        | Exploratory Data Analysis (Initial) | Broad exploration: stats, distributions, correlations, missing data |
|Step 4        | Data Cleaning           | Handle missing values, outliers, duplicates.            |
|Step 5        | Feature Engineering & Transformation            | Feature creation, scaling, encoding, selection                         |
|        | Subset Validation EDA            | Re-examine chosen features: check distributions, multicollinearity, relationships                      |               
|Step 6        | Modeling                  | Select algorithm(s), train models, tune hyperparameters                           |
|Step 7        | Evaluation                   | Validate using metrics and backtesting       |

### (1) Load Data
We will retrieve the adjusted closed price of SPY from locally stored data.

In [ ]:
# Load file
df = pd.read_csv('data/spy.csv', index_col=0, parse_dates=True)

# Calculate returns
df['Returns'] = np.log(df['Adj Close']).diff()
df = df["2010":]

# Verify the output
df

### (2) EDA of Original dataset

In [ ]:
# Descriptive statistics
df.describe().T

### (3) Cleaning & Imputation

Data is already cleaned. No further processing or imputation required.


In [ ]:
# Check for missing values
df.isnull().sum()

### (4) Feature Engineering
Features, also known as independent variables, are used to predict the value of the target variable (or label). In this step, we will create both features and the target variable from the raw dataset.

Feature engineering involves deriving new, meaningful features from the existing data. This process can significantly improve model performance and provide deeper insights into the underlying patterns within the data.

In [ ]:
# Create features (predictors) list
features_list = []
for r in range(10, 65, 5):
    df['Ret_'+str(r)] = df.Returns.rolling(r).sum()
    df['Std_'+str(r)] = df.Returns.rolling(r).std()
    features_list.append('Ret_'+str(r))
    features_list.append('Std_'+str(r))

# Drop NaN values
df.dropna(inplace=True)

#### (a) Feature Specification

In [ ]:
# Convert to NumPy
X = df.drop(['Open', 'High', 'Low', 'Close', 'Adj Close', 'Returns'],axis=1)
X.head(2)

#### (b) Target or Label Definition

Label or the target variable is also known as the dependent variable. Here, the target variable is whether the underlying price will close up or down on the next trading day. If the tomorrow’s closing price is greater than today’s closing price, then we will buy the underlying, else do nothing.

We assign a value of +1 for the buy signal and 0 otherwise to target variable. The target can be described as : <br><br>

$$
y_t =\begin{cases}
    1, & \text{if $p_{t+1} > 0.995 * p_{t}$}\\
    0, & \text{if $p_{t+1}  \space \text{Otherwise}$}
    \end{cases}
$$

whre, $p_{t}$ is the current closing price of the underlying and $p_{t+1}$ is the 1-day forward closing price of the underlying.

In [ ]:
# Define Target
# y = df['Label']
y = np.where(df['Adj Close'].shift(-1)>0.995 * df['Adj Close'],1,0)
y

In [ ]:
# label count
class_labels = np.bincount(y)
class_labels

> _**Note**: Feature engineering and artifact logging are not explicitly demonstrated here but are expected to be incorporated in a real-world ML workflow._

### (5) Boosting Ensemble

**Base Model**: We now build a base model with default parameters.

**Train-Test Split**

In [ ]:
# Splitting the datasets into training and testing data.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Output the train and test data size
print(f"Train and Test Size {len(X_train)}, {len(X_test)}")

**Fit Model**

In [ ]:
# Scale and fit the classifier model

# For binary or multiclass classification
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

base_model = XGBClassifier(
    verbosity=0,
    eval_metric='logloss'
)

base_model.fit(
    X_train,
    y_train,
    sample_weight=sample_weights
)


**Predict Model**

In [ ]:
# Predicting the test dataset
y_pred = base_model.predict(X_test)

# Predict Probabilities
y_proba = base_model.predict_proba(X_test)

In [ ]:
# Accuracy Scores
acc_train = accuracy_score(y_train, base_model.predict(X_train))
acc_test = accuracy_score(y_test, y_pred)

print(f'Train Accuracy: {acc_train:0.4}, Test Accuracy: {acc_test:0.4}')

In [ ]:
# Balanced Accuracy Scores
bal_acc_train = balanced_accuracy_score(y_train, base_model.predict(X_train))
bal_acc_test = balanced_accuracy_score(y_test,y_pred)

print(f'Train Balanced Accuracy: {bal_acc_train:0.4}, Test Balanced Accuracy: {bal_acc_test:0.4}')

In [ ]:
# Display confussion matrix
disp_cm = ConfusionMatrixDisplay.from_estimator(
        base_model,
        X_test,
        y_test,
        display_labels=base_model.classes_,
        cmap=plt.cm.Blues
    )
disp_cm.ax_.set_title('Confusion matrix')
plt.show()

In [ ]:
# Classification Report
print(classification_report(y_test, y_pred))

In [ ]:
# Display ROCCurve
disp_roc = RocCurveDisplay.from_estimator(
            base_model,
            X_test,
            y_test,
            name='XGBoost')

disp_roc.ax_.set_title('ROC Curve')
plt.plot([0,1], [0,1], linestyle='--')
plt.show()

In [ ]:
# Display PR Curve
disp_pr = PrecisionRecallDisplay.from_estimator(
    base_model,
    X_test,
    y_test,
    name='XGBoost')

disp_pr.ax_.set_title('Precision-Recall Curve')
plt.show()

In [ ]:
# W&B Run: Base Model
run = wandb.init(
    project="xgboost-demo",
    name="base-model",
    job_type="baseline",
    config=base_model.get_params(),
    reinit=True,
)

cr = classification_report(y_test, y_pred, output_dict=True)
cr_rows = [[label, m["precision"], m["recall"], m["f1-score"], int(m["support"])]
           for label, m in cr.items() if isinstance(m, dict)]

# Summary metrics
run.summary["train/accuracy"]     = float(acc_train)
run.summary["test/accuracy"]      = float(acc_test)
run.summary["train/bal_accuracy"] = float(bal_acc_train)
run.summary["test/bal_accuracy"]  = float(bal_acc_test)
run.summary["test/roc_auc"]      = float(roc_auc_score(y_test, y_proba[:, 1]))
run.summary["test/f1"]           = float(f1_score(y_test, y_pred))
run.summary["test/precision"]    = float(precision_score(y_test, y_pred))
run.summary["test/recall"]       = float(recall_score(y_test, y_pred))

wandb.log({
    "classification_report": wandb.Table(
        data=cr_rows,
        columns=["class", "precision", "recall", "f1-score", "support"]),
    "plots/confusion_matrix": wandb.plot.confusion_matrix(
        y_true=y_test, preds=y_pred, class_names=["0", "1"]),
    "plots/roc": wandb.plot.roc_curve(y_test, y_proba, labels=["0", "1"]),
    "plots/pr":  wandb.plot.pr_curve(y_test, y_proba, labels=["0", "1"]),
})

# from wandb.sklearn import plot_classifier
# This single line generates the confusion matrix, ROC, PR, and more automatically
# wandb.log({"eval_plots": plot_classifier(base_model, X_train, X_test, y_train, y_test, y_pred, y_proba, labels=["0", "1"])})

run.finish()
print("Base model logged to W&B")


### (6) Hyperparameter Tuning

Hyper-parameters are parameters that are not directly learnt within estimators. They are passed as arguments to the constructor of the estimator classes (Classifier in this case). It is possible and recommended to search the hyper-parameter space for the best cross validation score. Any parameter provided when constructing an estimator may be optimized in this manner.

We will tune the hyperparameters to select the best score by TimeSeriesSplit cross-validation. This is a variation of KFold. In the kth split, it returns first k folds as train set and the (k+1)th fold as test set. Unlike standard cross-validation methods, successive training sets are supersets of those that come before them.

The Three search methods used frequently are:

- **Grid Search**: This method exhaustively tries every combination of hyperparameters from a predefined set of values. While it guarantees finding the best combination within the defined grid, it can be computationally very expensive, especially with many hyperparameters or large ranges.

- **Random Search**: This method samples hyperparameters from a specified distribution for a fixed number of iterations. It's often more efficient than grid search in high-dimensional spaces because it can explore more of the parameter space, rather than being stuck on a grid.

- **Bayesian Search**: This is a more sophisticated method that builds a probabilistic model of the objective function (e.g., model performance) based on past evaluation results. It then uses this model to intelligently choose the next set of hyperparameters to try, aiming to find the optimum in fewer iterations. It's generally more efficient than random or grid search for complex, expensive-to-evaluate functions.

#### (a) XGBoost's hyper-parameter
XGBoost has plethora of tuning parameters including the regulatization parameters and some of the most common hyperparameters are:

* `learning rate`: step size shrinkage used in update to prevents overfitting. Range is [0,1].
* `max_depth`: maximum depth of a tree.
* `colsample_bytree`: percentage of features used per tree. High value can lead to overfitting.
* `min_child_weight`: minimum sum of instance weight needed in a child.
* `gamma`: minimum loss reduction required to make a further partition on a leaf node of the tree. Large gamma will lead to more conservative algorithm.
* `lambda`: L2 regularization term on weights. Increasing this value will make model more conservative. Normalised to number of training examples. [parameter for linear booster].

Refer [here](https://xgboost.readthedocs.io/en/latest/parameter.html#general-parameters) for complete list of tuning parameters.

In [ ]:
# Timeseries Cross Validation 2-split Demonstration
tscv = TimeSeriesSplit(n_splits=2, gap=1)
for train, test in tscv.split(X):
    print(f"Train: {train}, Test: {test}")

In [ ]:
# Cross-validation
tscv = TimeSeriesSplit(n_splits=5, gap=1)

In [ ]:
# Get params list
base_model.get_params()

#### (b) W&B Sweeps

W&B Sweeps provide a powerful and flexible way to automate hyperparameter optimization. Instead of a fixed number of parameter settings like in RandomizedSearchCV, W&B Sweeps allows you to define a search strategy (e.g., random, grid, or Bayesian search) and manage the entire tuning process, including logging, visualizing, and comparing runs. It integrates seamlessly with your training code, providing real-time insights into your hyperparameter search.

**Bayesian optimization** is a strategy used in W&B Sweeps for hyperparameter tuning. Unlike random search or grid search, which explore the search space blindly, Bayesian optimization builds a probabilistic model (a surrogate model) of the objective function (e.g., roc_auc_mean) and uses it to intelligently select the next set of hyperparameters to evaluate. This approach often finds optimal hyperparameters in fewer iterations by focusing on promising regions of the search space.

In [ ]:
# W&B Sweep config — replaces RandomizedSearchCV
sweep_config = {
    "method": "bayes",
    "metric": {"name": "cv/roc_auc_mean", "goal": "maximize"},
    "parameters": {
        "learning_rate":    {"values": [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]},
        "max_depth":        {"values": [3, 4, 5, 6, 8, 10, 12, 15]},
        "min_child_weight": {"values": [1, 3, 5, 7]},
        "gamma":            {"values": [0.0, 0.1, 0.2, 0.3, 0.4]},
        "colsample_bytree": {"values": [0.3, 0.4, 0.5, 0.7]},
        "n_estimators":     {"values": [100, 200, 300]},
    },
}

In [ ]:
# Define sweep training function
sweep_counter = 0

def sweep_train():
    global sweep_counter
    sweep_counter += 1

    run = wandb.init(name=f"sweep-{sweep_counter}")
    cfg = wandb.config

    model = XGBClassifier(
        learning_rate=cfg.learning_rate,
        max_depth=cfg.max_depth,
        min_child_weight=cfg.min_child_weight,
        gamma=cfg.gamma,
        colsample_bytree=cfg.colsample_bytree,
        n_estimators=cfg.n_estimators,
        verbosity=0,
        eval_metric='logloss',
    )

    # CV score with same TimeSeriesSplit
    # sklearn >=1.4 renamed fit_params to params; use params for newer versions
    cv_scores = cross_val_score(
        model, X_train, y_train, cv=tscv,
        scoring='roc_auc',
        params={'sample_weight': sample_weights}
    )
    run.summary["cv/roc_auc_mean"] = float(cv_scores.mean())
    run.summary["cv/roc_auc_std"]  = float(cv_scores.std())

    # Full train + test eval
    model.fit(X_train, y_train, sample_weight=sample_weights)
    y_p = model.predict(X_test)
    y_pb = model.predict_proba(X_test)

    run.summary["test/accuracy"]     = float(accuracy_score(y_test, y_p))
    run.summary["test/bal_accuracy"] = float(balanced_accuracy_score(y_test, y_p))
    run.summary["test/roc_auc"]      = float(roc_auc_score(y_test, y_pb[:, 1]))
    run.summary["test/f1"]           = float(f1_score(y_test, y_p))

    run.finish()

# Launch sweep — 50 trials
sweep_id = wandb.sweep(sweep_config, project="xgboost-demo")
wandb.agent(sweep_id, function=sweep_train, count=50)


In [ ]:
# Retrieve best parameters from the sweep
api = wandb.Api()
sweep = api.sweep(f"{api.default_entity}/xgboost-demo/{sweep_id}")
best_run = sweep.best_run()
best_params = {k: v for k, v in best_run.config.items() if not k.startswith("_")}

print("\nBest parameters found by W&B Sweep:")
for k, v in best_params.items():
    print(f"  {k}: {v}")
print(f"Best cv/roc_auc_mean: {best_run.summary.get('cv/roc_auc_mean', 'N/A'):.4f}")


**Tuned Model**: Let's now train and predict the model with the best search parameter

In [ ]:
# Create tuned model using best parameters from sweep
tuned_model = XGBClassifier(
    **best_params,
    verbosity=0,
    eval_metric='logloss',
)

# Fit with evaluation tracking
tuned_model.fit(
    X_train, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=False,
)

# --- W&B Run 3: Tuned Model ---
# Link back to the sweep and best run for traceability
run = wandb.init(
    project="xgboost-demo",
    name="tuned-model",
    job_type="tuned",
    config={
        **best_params,
        "sweep_id": sweep_id,
        "best_run_id": best_run.id,
        "best_run_name": best_run.name,
        "best_cv_roc_auc": best_run.summary.get("cv/roc_auc_mean"),
    },
    reinit=True,
)

# Log per-round logloss — step-based charts work perfectly here
evals = tuned_model.evals_result()
for i, (train_loss, val_loss) in enumerate(
    zip(evals['validation_0']['logloss'], evals['validation_1']['logloss'])
):
    wandb.log({"loss/train": train_loss, "loss/val": val_loss}, step=i)

print("Tuned model training completed successfully!")


In [ ]:
# Return the evaluation results
# evals_result = tuned_model.evals_result()
# evals_result

In [ ]:
# Cross validation score with tuned model
cv_scores = cross_val_score(tuned_model, X_train, y_train, cv=tscv, scoring='f1')
print(f'Mean CV Score: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})')

**Predict Model**

In [ ]:
# Predicting the test dataset
y_pred = tuned_model.predict(X_test)
y_proba = tuned_model.predict_proba(X_test)

# Measure Accuracy
acc_train = accuracy_score(y_train, tuned_model.predict(X_train))
acc_test = accuracy_score(y_test, y_pred)

print(f'\n Training Accuracy \t: {acc_train :0.4} \n Test Accuracy \t\t: {acc_test :0.4}')

# Log to active W&B run
run.summary["test/accuracy"] = float(acc_test)
run.summary["train/accuracy"] = float(acc_train)


In [ ]:
bal_acc_train = balanced_accuracy_score(y_train, tuned_model.predict(X_train))
bal_acc_test = balanced_accuracy_score(y_test, y_pred)

print(f'Train Balanced Accuracy: {bal_acc_train:0.4}, Test Balanced Accuracy: {bal_acc_test:0.4}')

run.summary["train/bal_accuracy"] = float(bal_acc_train)
run.summary["test/bal_accuracy"]  = float(bal_acc_test)
run.summary["test/roc_auc"]      = float(roc_auc_score(y_test, y_proba[:, 1]))
run.summary["test/f1"]           = float(f1_score(y_test, y_pred))


In [ ]:
# --- Tuned Model: Evaluation + W&B ---

# Confusion Matrix
disp = ConfusionMatrixDisplay.from_estimator(
        tuned_model, X_test, y_test,
        display_labels=tuned_model.classes_, cmap=plt.cm.Blues)
disp.ax_.set_title('Confusion matrix')
plt.show()

# Classification Report
print(classification_report(y_test, y_pred))

# ROC Curve
disp_roc = RocCurveDisplay.from_estimator(
            tuned_model, X_test, y_test, name='Tuned XGBoost')
disp_roc.ax_.set_title('ROC Curve')
plt.plot([0,1], [0,1], linestyle='--')
plt.show()

# Log all to active W&B run
cr = classification_report(y_test, y_pred, output_dict=True)
cr_rows = [[label, m["precision"], m["recall"], m["f1-score"], int(m["support"])]
           for label, m in cr.items() if isinstance(m, dict)]
wandb.log({
    "classification_report": wandb.Table(
        data=cr_rows,
        columns=["class", "precision", "recall", "f1-score", "support"]),
    "plots/confusion_matrix": wandb.plot.confusion_matrix(
        y_true=y_test, preds=y_pred, class_names=["0", "1"]),
    "plots/roc": wandb.plot.roc_curve(y_test, y_proba, labels=["0", "1"]),
    "plots/pr":  wandb.plot.pr_curve(y_test, y_proba, labels=["0", "1"]),
})


### (7) Evaluation

#### (a) Feature Importance

Feature Importance refers to techniques that calculate a score for all the input features for a given model where the scores represent the “importance” of each feature. It is calculated as the decrease in node impurity weighted by the probability of reaching that node. The node probability can be calculated by the number of samples that reach the node, divided by the total number of samples. The higher the value the more important the feature.

The Gain is the most relevant attribute to interpret the relative importance of each feature.

In [ ]:
# Plot the feature importance of the tuned model
plot_importance(tuned_model, importance_type='weight', title='Tuned Model Feature Importance', show_values=False)
plt.tight_layout()
plt.grid(True, alpha=0.3)
plt.show()

# Log feature importance to W&B
imp = tuned_model.get_booster().get_score(importance_type='gain')
imp_table = wandb.Table(
    data=sorted(imp.items(), key=lambda x: -x[1]),
    columns=["feature", "gain"],
)
wandb.log({
    "feature_importance": wandb.plot.bar(
        imp_table, "feature", "gain",
        title="Feature Importance (Gain)"),
})


Importance type can be either of the following:

* weight: the number of times a feature is used to split the data across all trees.
* gain: the average gain across all splits the feature is used in.
* cover: the average coverage across all splits the feature is used in.
* total_gain: the total gain across all splits the feature is used in.
* total_cover: the total coverage across all splits the feature is used in.

By default, `feature_importances_` rank features based on the average gain across all splits. This can be changed using the `plot_importance` method.

In [ ]:
# The Gain is the most relevant attribute to interpret the relative importance of each feature.
# plot_importance?

In [ ]:
# Feature importance by gain
plot_importance(tuned_model, importance_type='gain', show_values=False)
plt.show()

# End the W&B run for tuned model
run.finish()
print("Tuned model logged to W&B")


**Plot Tree**

In [ ]:
## Tree Visualization
# change tree number to see the corresponding plot
to_graphviz(tuned_model, tree_idx=2, rankdir='UT')

The values shown in leaf nodes are the raw score contributions. Each leaf node shows something like leaf=x where x is the raw score. Following a path from root to leaf, if a sample ends up in a leaf with value x. For binary classification, the probability of class 1 is $\sigma(x)$. The final prediction is made by summing up contributions from all trees and then applying logistic (sigmoid) function.

_raw_score = base_score + tree1_leaf_value + tree2_leaf_value + ... + treeN_leaf_value_

Each sample passes through an ensemble of decision trees, summing the additive contributions of shallow learners via logit aggregation.

#### (b) Tree Analysis

1. Each tree will make a similar decision path based on the features of the input and give one raw score from its own leaf node.

1. Add up raw scores from all 300 trees

1. $  logit =  \sum_{t=0}^{299} f(x) $

    let's say the total logit from all 300 trees is `0.3519`.

1. Apply the sigmoid to get probability

    $ probability = \frac{1}{1+e^{-0.3519}} \approx 0.587 $

1. If the probability above 0.5, then assign $x$ to Class 1, else Class 0

1. Repeat the above for each sample, in this case for 508 days

## Section 3: Also Read

📌 [Quantmod](https://kannansingaravelu.com/quantmod/)

📌 [Scikit-learn](https://scikit-learn.org/stable/index.html)

📌 [XGBoost](https://xgboost.readthedocs.io/en/latest/index.html)

📌 [XGBoost (Clone)](https://xgboost-clone.readthedocs.io/en/latest/python/python_intro.html)

📌 [Weights & Biases](https://docs.wandb.ai/)  

---
[Kannan Singaravelu](https://www.linkedin.com/in/kannansi)